# 3D Fluorescence LIVE/DEAD Segmentation (v1.2.1)


> **Python 3.10, 3.11, 3.12, 3.13, or 3.14.** This notebook and `requirements.txt` support all five; the pins that differ by version are handled automatically via markers in `requirements.txt` (see its header comment and README.md's Prerequisites section). Cell 1 below prints which Python version it detected and whether it's a supported one. Install with `uv pip install -r requirements.txt --override overrides.txt` (`uv` required -- see README.md for why).
>
> **Python 3.14 limitations:** TensorFlow has no Python 3.14 wheels, so TensorFlow, `csbdeep` and `stardist` are not installed there and **StarDist is unavailable**. If `trig_stardist = True` is set on 3.14, Cell 3 prints a warning and automatically sets it back to `False`. Likewise, `aicspylibczi` (the Zeiss reader) has no 3.14 wheels, so **`.czi` files can't be opened on 3.14** (Cell 4 raises a clear error) — use Python 3.10–3.13 for `.czi`, or export to OME-TIFF. Watershed, Cellpose 3D and all other file formats work normally.

---


This notebook is a **generic template** for 3D fluorescence segmentation and quantification of LIVE/DEAD-stained samples, with a **binary LIVE/DEAD call for every cell**.

It follows the nuclei notebook (`Fluo_3D_nuc_seg_v1.6.2`) step by step. The difference is the stain table: each marker is set as **nuclear** (e.g. ethidium homodimer, propidium iodide, a Hoechst counterstain) or **cytoplasmic** (e.g. calcein), and everything else follows from that:
- the nuclear markers make up the nuclei (Cell 17), the cytoplasmic markers the cell bodies (Cell 18), combined into one label per cell;
- each marker is measured in its own compartment (nucleus or whole cell) and every cell is called LIVE or DEAD (Cell 20).

**Before running the workflow:**
1. Set the file path, ROI and experiment name in **Cell 3**.
2. Run Cell 4, fill in the stain table (channel, color, marker and localization of LIVE, DEAD and any extra stain) and click **Confirm**.
3. Start with the default processing settings and tune only the parameters that need adjusting for your dataset.

**Segmentation methods available:**
- **Watershed** (default) — classical multi-scale erosion + EDT peaks, for the nuclei and the cell bodies. Splitting of touching objects is automatic (built-in first pass, then size-based iterative re-splitting — see Cell 3, section 7).
- **Cellpose 3D** — deep-learning model; handles touching objects natively. Set `trig_cellpose = True` for the nuclei and/or `trig_cellpose_cyto = True` for the cell bodies.
- **StarDist** — 2D-to-3D translation model for the nuclei. Set `trig_stardist = True`. *Not available on Python 3.14 (no TensorFlow wheels) — automatically disabled with a warning.*

---

# 1. Import Libraries and Helpers

Load the Python packages and the helper functions.

**Cell 1** (Import libraries)

Loads all required packages (napari, scipy, scikit-image, ...) and prints the detected Python version.

In [ ]:
from helpers.notebook_setup_helpers import load_nuclei_notebook_setup
globals().update(load_nuclei_notebook_setup())

**Cell 2** (Load helper functions)

Loads the functions from `helpers/notebook_helpers.py`. Re-run it after editing that file (no kernel restart needed).

In [ ]:
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 2. Inputs and Setup

Your settings and the stain table. For a new dataset, these are the only cells you need to edit.

**Cell 3** (Settings)

Input file (or a folder of images for **batch mode**), ROI, experiment name, object sizes, segmentation method, LIVE/DEAD call and size refinement. Each setting is explained next to it.

- **Sizes:** `nuclei_diameter` is used to segment the nuclear markers, `cell_diameter` to segment the cytoplasmic markers (whole cells).
- **Segmentation method:** leave `trig_cellpose` and `trig_stardist` both `False` for watershed nuclei, and `trig_cellpose_cyto = False` for watershed cell bodies. StarDist is not available on Python 3.14 (it is switched off with a warning).
- **LIVE/DEAD call:** `min_positive_fraction` sets when a cell counts as positive for a marker, `ld_rule` how the LIVE and DEAD markers decide the call (see Cell 20).
- **Size refinement:** after segmentation, labels much bigger than a typical nucleus (or cell, for the cell bodies) are split again with increasingly aggressive settings, and small fragments are merged into a touching neighbour when they look like the same object. Labels are never deleted.

In [ ]:
# --- 1. Input -----------------------------------------------------------------
# Image file (.nd2, .tif/.ome.tif, .czi, .lif, .oib) or a FOLDER of images
# (batch mode: outputs go to '<folder>_output/<image name>/').
input_file = 'path/to/your_image_file.nd2'

# ROI = [x0, x1, y0, y1, z0, z1]; 0 = keep the full range.
ROI = [0, 0, 0, 0, 0, 0]
# True = drag the ROI (X/Y) in a napari window before loading.
interactive_roi = False

# --- 2. Experiment -------------------------------------------------------------
name_setup = 'your_experiment_name'  # prefix of the saved setup/output files
use_setup = True            # reuse the saved stain table and contrast/gamma
automatic_contrast = False  # True = pick contrast automatically (no napari window)

# --- 3. Sizes -----------------------------------------------------------------
nuclei_diameter = 10  # um, nucleus (segmentation of the nuclear markers)
cell_diameter = 30    # um, whole cell (segmentation of the cytoplasmic markers)
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]

# --- 4. Segmentation method ---------------------------------------------------
# All False = watershed (default).
trig_cellpose = False  # Cellpose 3D for the nuclei (GPU recommended)
trig_stardist = False  # StarDist for the nuclei (not available on Python 3.14)
trig_stardist = resolve_stardist_flag(trig_stardist)
trig_cellpose_cyto = False  # Cellpose 3D for the cell bodies (cytoplasmic markers)

# --- 5. Quantification ----------------------------------------------------------
multilabel = True            # multi-marker combinations in the summary
aggregate_grow_factor = 2.0  # growth used to detect cell aggregates

# --- 6. LIVE/DEAD call ----------------------------------------------------------
# Positive for a marker = the marker covers at least this fraction of the cell's
# compartment (nucleus for nuclear, whole cell for cytoplasmic markers); 0 = any voxel.
min_positive_fraction = 0.2
# "dead_priority": DEAD-positive = DEAD, otherwise LIVE.
# "coverage": positive for both or neither = the marker covering more of its compartment.
ld_rule = "dead_priority"

# --- 7. Size refinement (automatic splitting/merging, nuclei and cell bodies) ---
size_reference = "blend"  # reference size: "blend", "input" or "detected"
iterative_split = True    # re-split labels > oversize_factor x reference
oversize_factor = 2.0
max_split_iterations = 3  # 1-3 escalating split levels
merge_undersized = True   # merge fragments < undersize_factor x reference
undersize_factor = 0.5

**Cell 4** (Load image and stain table)

1. Loads the image and reads voxel size and metadata (large files are read in chunks automatically). With `interactive_roi = True`, a napari window opens first to drag the ROI. In batch mode nothing is loaded: the channels of all images are checked instead, and any image with a different channel list is flagged.
2. Prints the channel list.
3. Shows the **LIVE/DEAD stain table**. For LIVE and DEAD choose the channel (`no` = not used) and color, type the marker name and choose its **localization**:
   - **Nuclear** for stains of the nucleus (e.g. ethidium homodimer, propidium iodide, a Hoechst/DAPI counterstain);
   - **Cytoplasmic** for stains that fill the cell (e.g. calcein).

   Extra stains (e.g. a nuclear counterstain of all cells) can be added with **Add stain**: they help the segmentation and are quantified, but don't enter the LIVE/DEAD call. At least one of LIVE and DEAD needs a channel. Click **Confirm**: the table is saved to `<name_setup>_ld_stain_dict.json` and reused next time (`use_setup = True`).

In [ ]:
batch_mode = is_batch_input(input_file)
file_meta = None

if batch_mode:
    print(f"Batch mode: {len(list_image_files(input_file))} image(s) will be "
          f"processed by Cell 5.")
else:
    if interactive_roi:
        ROI = select_roi_interactively(input_file, ROI, napari_module=napari)
    (meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
     cyto_factor, PCM_factor, zoom_factors, used_lazy_loading) = initialize_dataset(
        input_file, ROI,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=scale_factor,
        zoom_factors=zoom_factors,
    )

channel_list = print_channel_list(input_file, file_meta=file_meta)
stain_editor = edit_ld_stain_dict(channel_list, name_setup=name_setup, use_saved=use_setup)

**Cell 5** (Read the stain table and run the batch)

Reads the confirmed stain table: `stain_dict` and each marker's `localization`. If it isn't confirmed yet, it stops with a message: click **Confirm** in Cell 4, then run again from here.

In **batch mode** it then processes every image in the folder with the settings above (no viewers or plots) and saves each image's outputs to `<folder>_output/<image name>/`, plus `live_dead_batch_summary.xlsx` with the cell counts and viability of every image. The image-processing parameters for the batch are set in this cell. When the batch finishes, execution stops: the cells below are for single-file mode only.

In [ ]:
stain_dict, localization = stain_editor.result()
nuclear_conditions = [c for c, loc in localization.items() if loc == 'nuclear']
print(f"stain_dict = {stain_dict}")
print(f"localization = {localization}")

# --- Batch mode only (input_file is a folder) --------------------------------
# Image processing (set here because the tuning cells don't run in batch mode).
batch_sigma = 0.5                # smoothing strength: 0 = none, higher = smoother (noisy data)
batch_num_plateaus = 2           # contrast boost: raise to 3-4 for dim, low-contrast channels
batch_plateau_factor = 0.7       # contrast boost limit (0.5-1.0): lower = gentler, less noise amplified
batch_threshold_method = 'otsu'  # 'otsu', 'median' (sparse) or 'huang' (dim)

# Optional (slow) 3D exports, per image.
batch_export_vtk = True  # 3D volumes for ParaView
batch_export_stl = True  # 3D surface meshes per marker
batch_export_fea = True  # finite-element mesh (Abaqus .inp)

if batch_mode:
    batch_summary = run_ld_batch_folder(
        input_file,
        roi_coords=ROI,
        stain_dict=stain_dict,
        localization=localization,
        name_setup=name_setup,
        use_setup=use_setup,
        automatic_contrast=automatic_contrast,
        interactive_roi=interactive_roi,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=scale_factor,
        zoom_factors=zoom_factors,
        trig_cellpose=trig_cellpose,
        trig_stardist=trig_stardist,
        trig_cellpose_cyto=trig_cellpose_cyto,
        multilabel=multilabel,
        aggregate_grow_factor=aggregate_grow_factor,
        min_positive_fraction=min_positive_fraction,
        ld_rule=ld_rule,
        size_reference=size_reference,
        iterative_split=iterative_split,
        oversize_factor=oversize_factor,
        max_split_iterations=max_split_iterations,
        merge_undersized=merge_undersized,
        undersize_factor=undersize_factor,
        sigma=batch_sigma,
        num_plateaus=batch_num_plateaus,
        plateau_factor=batch_plateau_factor,
        threshold_method=batch_threshold_method,
        export_vtk=batch_export_vtk,
        export_stl=batch_export_stl,
        export_fea=batch_export_fea,
        settings=settings,
        napari_module=napari,
        progress=tqdm,
    )
    display(batch_summary)
    # Stop 'Run All' here: the remaining cells are for single-file mode.
    raise BatchCompleted("Batch processing finished -- the remaining cells are for single-file mode only.")
else:
    print("Single-file mode -- continue with the next cells.")

# 3. Preparation and Preview

Prepare the image stack and set the display contrast/gamma of each channel.

**Cell 6** (Prepare the image stack)

Converts the image, builds the stain table (`stain_df`) and opens a napari viewer with each channel. Channels not used in the stain table are dropped.

In [ ]:
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari, r_xyz=(r_X, r_Y, r_Z), progress=tqdm,
)

**Cell 7** (Contrast and gamma)

Reuses the saved `<name_setup>_setup.csv` if it exists (`use_setup = True`). Otherwise a napari window opens to adjust contrast/gamma per channel — or, with `automatic_contrast = True`, the contrast limits are picked automatically from each channel's histogram (gamma stays 1.0).

In [ ]:
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    automatic_contrast=automatic_contrast,
    settings=settings,
    napari_module=napari,
    r_xyz=(r_X, r_Y, r_Z),
    progress=tqdm,
)

**Cell 8** (Review the stain settings)

Shows the complete stain settings table. Check the contrast limits and gamma before continuing.

In [ ]:
stain_complete_df

# 4. Image Processing and Segmentation

Run the cells in order: each step feeds the next and shows a histogram so you can follow its effect. Only Cells 13-15 have parameters to tune.

**Cell 9** (Normalize)

Scales each channel to 0-255.

In [ ]:
im_final_stack = run_normalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 10** (Resample)

Resamples the image to the same voxel size in X, Y and Z.

In [ ]:
im_final_stack, r_zX, r_zY, r_zZ = run_resample(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    zoom_factors=zoom_factors,
    meta=meta,
)

**Cell 11** (Denoise)

Median filter to remove noise.

In [ ]:
im_final_stack = run_denoise(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 12** (Apply contrast and gamma)

Applies the contrast/gamma of each channel set in Cell 7.

In [ ]:
im_final_stack = run_contrast_gamma(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 13** (Smoothing)

Gaussian smoothing. Raise `sigma` for noisy data; set it to 0 to skip smoothing.

In [ ]:
sigma = 0.5  # smoothing strength: 0 = none, 0.5 = mild, 1-2 = strong (noisy data)

im_final_stack = run_smooth(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    sigma=sigma,
)

**Cell 14** (Contrast enhancement)

Histogram equalization to boost contrast before thresholding. The defaults suit most data.

In [ ]:
num_plateaus = 2      # contrast boost: raise to 3-4 for dim, low-contrast channels
plateau_factor = 0.7  # contrast boost limit (0.5-1.0): lower = gentler, less noise amplified

im_final_stack = run_equalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    num_plateaus=num_plateaus,
    plateau_factor=plateau_factor,
)

**Cell 15** (Thresholding)

Separates signal from background in every channel (`run_threshold`): a global threshold combined with a local one and a background-statistics one. `threshold_method` picks the global part. The nuclear markers are thresholded at the scale of a nucleus, the cytoplasmic ones at the scale of a cell. The histogram marks, per channel, the global (dashed orange) and final (solid red) thresholds.

These masks are used everywhere downstream:
- **nuclear** markers → merged into the mask the nuclei are segmented from (Cell 17);
- **cytoplasmic** markers → merged into the mask the cell bodies are segmented from (Cell 18);
- every marker → which cells are positive for it (Cell 20).

In [ ]:
threshold_method = 'otsu'  # 'otsu' (clear signal/background), 'median' (sparse) or 'huang' (dim/diffuse)

im_final_stack = run_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    threshold_method=threshold_method,
    nuclear_conditions=nuclear_conditions,
    progress=tqdm,
)

**Cell 16** (Export histograms)

Saves the histograms of every processing step, plus a Parameters sheet (including the LIVE/DEAD settings in `ld_settings`, also written to the Excel report of Cell 34), to `<file>_histograms.xlsx`.

In [ ]:
ld_settings = {
    'Localization': localization,
    'Nuclei diameter (um)': nuclei_diameter,
    'Cell diameter (um)': cell_diameter,
    'Use Cellpose (nuclei)': trig_cellpose,
    'Use StarDist (nuclei)': trig_stardist,
    'Use Cellpose (cell bodies)': trig_cellpose_cyto,
    'Min positive fraction': min_positive_fraction,
    'LIVE/DEAD rule': ld_rule,
    'Size reference': size_reference,
    'Iterative split': iterative_split,
    'Oversize factor': oversize_factor,
    'Max split iterations': max_split_iterations,
    'Merge undersized': merge_undersized,
    'Undersize factor': undersize_factor,
}

output_path = export_channel_histograms(
    im_final_stack, stain_complete_df, input_file,
    ROI_print=ROI_print, lazy_loading_used=used_lazy_loading,
    name_setup=name_setup,
    nuclei_diameter=nuclei_diameter, cell_diameter=cell_diameter,
    cyto_factor=cyto_factor, PCM_factor=PCM_factor,
    zoom_factors=zoom_factors, scale_factor=scale_factor,
    trig_cellpose=trig_cellpose, trig_stardist=trig_stardist,
    trig_cellpose_cyto=trig_cellpose_cyto,
    multilabel=multilabel,
    sigma=sigma, num_plateaus=num_plateaus, plateau_factor=plateau_factor,
    threshold_method=threshold_method,
    aggregate_grow_factor=aggregate_grow_factor,
    extra_params=ld_settings,
    progress=tqdm,
)

**Cell 17** (Segment nuclei — nuclear markers)

The threshold masks of the markers set as **nuclear** are merged into one nuclei mask, which is segmented exactly like the NUCLEI channel of the nuclei notebook, with the method chosen in Cell 3 (watershed, Cellpose or StarDist). The watershed splits touching nuclei automatically, also where the nuclear signal dips between them.

The nuclei are then refined by size (Cell 3, section 7): fragments smaller than `undersize_factor` × reference are merged into a touching neighbour when they look like the same nucleus, nuclei bigger than `oversize_factor` × reference are split again with stronger settings. The **refinement recap** table lists what happened to every nucleus.

Without nuclear markers nothing is segmented here: each cell's nucleus is estimated in Cell 18.

In [ ]:
im_segmentation_stack = segment_ld_nuclei(
    im_final_stack,
    stain_df=stain_df,
    localization=localization,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    trig_cellpose=trig_cellpose,
    size_reference=size_reference,
    iterative_split=iterative_split,
    oversize_factor=oversize_factor,
    max_split_iterations=max_split_iterations,
    merge_undersized=merge_undersized,
    undersize_factor=undersize_factor,
    progress=tqdm,
)

**Cell 18** (Segment cells — cytoplasmic markers)

The threshold masks of the markers set as **cytoplasmic** are merged and segmented into cell bodies the same way, at the scale of `cell_diameter` (watershed and size refinement, or Cellpose `cyto3` with `trig_cellpose_cyto = True`). Nuclei and cell bodies are then combined into **one label per cell**:
- a nucleus lying (for at least half of its volume) in a cell body belongs to that cell; a body holding two or more nuclei is split between them;
- a nucleus outside every body is a cell without cytoplasmic signal (typical of a dead cell): the cell is its nucleus;
- a body without a stained nucleus (typical of a live cell when the nuclear marker only stains dead cells) gets an **estimated nucleus**: a sphere of `nuclei_diameter` at the centre of the cell. It gives the cell a position and the compartment where the nuclear markers are measured.

The cells are numbered 1…K, with the same number in the Cytoplasm (whole cell, nucleus included) and Nuclei layers, and the number of cells in each case is printed. Without cytoplasmic markers every cell is its stained nucleus.

In [ ]:
im_segmentation_stack, stain_complete_df = segment_ld_cells(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    localization=localization,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    trig_cellpose_cyto=trig_cellpose_cyto,
    size_reference=size_reference,
    iterative_split=iterative_split,
    oversize_factor=oversize_factor,
    max_split_iterations=max_split_iterations,
    merge_undersized=merge_undersized,
    undersize_factor=undersize_factor,
    progress=tqdm,
)

**Cell 19** (Segment PCM)

The pericellular matrix: a shell around each cell, as in the nuclei notebook. It is not used for the LIVE/DEAD call.

In [ ]:
im_segmentation_stack = segment_ld_pcm(
    im_segmentation_stack,
    cyto_factor=cyto_factor,
    PCM_factor=PCM_factor,
)

**Cell 20** (LIVE/DEAD call)

Each marker is measured in its own compartment: a **nuclear** marker in the cell's nucleus (stained or estimated), a **cytoplasmic** marker in the whole cell. A cell is **positive** for a marker when the marker's threshold mask (Cell 15) covers at least `min_positive_fraction` of that compartment (`0` = any voxel, as in the nuclei notebook).

Every cell is then called **LIVE** or **DEAD** from the LIVE and DEAD markers (`ld_rule`, Cell 3):
- `"dead_priority"` (default): DEAD-positive → DEAD, otherwise LIVE — a cell whose membrane lets the dead stain in is dead, even with some calcein left. Without a DEAD channel: LIVE-positive → LIVE, otherwise DEAD;
- `"coverage"`: positive for one marker only → that one; positive for both or neither → the marker covering more of its compartment.

The result is the per-cell table `ld_cells_df` (status, position, volume, and coverage, positivity and intensity of every marker) and, per marker, the label layer of its positive cells.

In [ ]:
im_segmentation_stack, ld_cells_df = classify_live_dead(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    localization=localization,
    r_zxyz=(r_zX, r_zY, r_zZ),
    min_positive_fraction=min_positive_fraction,
    ld_rule=ld_rule,
    progress=tqdm,
)
ld_cells_df

**Cell 21** (Detect aggregates)

Finds cell aggregates by growing each cell (`aggregate_grow_factor`) and checking which ones overlap.

In [ ]:
# stain_complete_df carries the CYTOPLASM row added in Cell 18 (the cells).
im_segmentation_stack = detect_aggregates(
    im_segmentation_stack,
    stain_df=stain_complete_df,
    aggregate_grow_factor=aggregate_grow_factor,
)

**Cell 22** (View the results)

Opens napari viewers with every processing step and the segmentation, as in the nuclei notebook (the "Thresh islands" layers show the connected regions of each channel's threshold mask, to check whether thresholding misses or breaks up cells). The segmentation viewer also shows the stained and the estimated nuclei, and the **LIVE** (green) and **DEAD** (red) cells of the call.

In [ ]:
viewer_0, viewer_1 = view_ld_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)

# 5. Quantification and Analysis

Measures every segmented cell and summarises the LIVE/DEAD populations. Marker intensity is reported two ways: **positive cells** (only cells positive for that marker) and **all cells** (every segmented cell).

**Cell 23** (Measure the cells)

Mean/std intensity of every marker in each cell (nucleus + cytoplasm + PCM, all cells), then marker overlap, intensity, volume and position of the positive cells of each marker (`labels_df`), as in the nuclei notebook. With `multilabel = True` the LIVE + DEAD row counts the double-positive cells.

In [ ]:
percell_mean_df, percell_std_df = compute_percell_marker_intensity_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    progress=tqdm,
)

labels_df = build_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zoom_factors,
    multilabel=multilabel,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
)

**Cell 24** (LIVE/DEAD summary)

The binary call: LIVE and DEAD counts, percentages and viability, and how each marker combination (LIVE only, DEAD only, both, neither) was called. Then the positivity of every marker and, for the LIVE and the DEAD cells, cell size, nucleus size and marker intensity.

In [ ]:
print_ld_summary(
    ld_cells_df,
    stain_df=stain_df,
)

**Cell 25** (Full table for export)

The marker measurements of Cell 23 in full (not truncated), used by the VTK export.

In [ ]:
labels_full_df = build_full_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zoom_factors,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
)

**Cell 26** (Per-cell report)

Intensity distribution (KDE) of each marker inside every positive cell, plotted and saved as a PDF report (one page per cell, centred on its nucleus).

In [ ]:
hist_data, intensity_ranges, fig, axes, x_grid = build_histogram_report(
    im_segmentation_stack, im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    progress=tqdm,
)

**Cell 27** (Spatial distribution)

Where the LIVE and DEAD cells are along X, Y and Z (number of cells, left axis), with the viability along each direction (% LIVE, dashed, right axis) — e.g. viability vs depth in a 3D construct.

In [ ]:
plot_ld_spatial_distributions(
    ld_cells_df,
    stain_complete_df=stain_complete_df,
    volume_shape=im_final_stack['Filtered image'].shape,
    r_zxyz=(r_zX, r_zY, r_zZ),
)

**Cell 28** (Size distribution)

Cell volume and (stained) nucleus volume of the LIVE and DEAD cells.

In [ ]:
plot_ld_size_distributions(
    ld_cells_df,
    stain_complete_df=stain_complete_df,
)

**Cell 29** (LIVE/DEAD call plot)

Left: LIVE and DEAD counts. Right: each cell's LIVE vs DEAD coverage (fraction of its compartment covered by the marker), coloured by the call, with `min_positive_fraction` as dashed lines — use it to check the positivity threshold. Saved as `<file>_live_dead.png`.

In [ ]:
plot_live_dead_classification(
    ld_cells_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
)

**Cell 30** (Marker intensity cloud)

Flow-cytometry-style plot: each cell's mean marker intensity (all cells) against its size, all markers together, with histograms on the sides. Saved as `<file>_marker_intensity.png`.

In [ ]:
plot_marker_intensity_clouds(
    percell_mean_df,
    stain_complete_df=stain_complete_df,
    im_segmentation_stack=im_segmentation_stack,
    r_xyz=(r_zX, r_zY, r_zZ),
    stain_df=stain_df,
    progress=tqdm,
    input_file=input_file,
)

# 6. Export

Writes the results to disk.

**Cell 31** (VTK volumes)

Labelled 3D volumes of nuclei, cells and PCM for ParaView. Each marker's all-cells intensity is included, so the meshes can be coloured by it.

In [ ]:
build_vtk_volumes(
    im_segmentation_stack,
    labels_full_df=labels_full_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
    progress=tqdm,
)

**Cell 32** (STL meshes)

One 3D surface mesh per marker (its positive cells).

In [ ]:
export_marker_stl(
    im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
    progress=tqdm,
)

**Cell 33** (Single-cell export, optional)

Set `nuc_3D_export = True` to export one cell (nucleus, cell and marker channels) as a VTK sub-volume for ParaView. When the cell runs, a text box asks for the cell's label number (the `Cell label` of `ld_cells_df`).

In [ ]:
nuc_3D_export = False  # True = export one cell (you'll be asked for its label number)

if nuc_3D_export:
    nuc_label = int(input("Enter cell label number to export: "))
    export_nucleus_vtk_crop(
        nuc_label=nuc_label,
        im_segmentation_stack=im_segmentation_stack,
        im_final_stack=im_final_stack,
        stain_df=stain_df,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        zoom_factors=zoom_factors,
    )

**Cell 34** (Excel export)

The LIVE/DEAD results in `<file>_live_dead.xlsx`: **Summary** (call, viability, LIVE/DEAD/ALL populations), **Cells** (one row per cell: status, position, size, and coverage, positivity and intensity of every marker) and **Setup** (stain table with each marker's localization, and `ld_settings`).

In [ ]:
output_path = export_live_dead_to_excel(
    input_file,
    original_stain_complete_df,
    ld_cells_df,
    parameters=ld_settings,
)

**Cell 35** (FEA mesh)

Abaqus `.inp` mesh of the cells (whole cells, one element set per cell) for finite element analysis.

In [ ]:
# Mesh the whole cells: many nuclei are estimated when the nuclear marker
# only stains the dead cells.
export_fea_mesh(
    {'Nuclei': im_segmentation_stack['Cytoplasm']},
    input_file=input_file,
    progress=tqdm,
)